# Round 2: data-centric experiments (one factor at a time)

**Runtime:** GPU (T4 is enough). Use **Runtime → Run all** on a fresh runtime. Any failure raises an exception that
stops the run and names the failing stage/step and its log file.

| Stage | Run | What changes vs its parent | Trains? | ~Time on T4 |
|---|---|---|---|---|
| setup | | verify bundle, install deps (`--use-pep517`), import check, unit tests | no | 4–6 min |
| R0 | `e2_mbert_v2_fmtA` | baseline: the existing round-1 E2 checkpoint, scored on the new v3 evaluation suite | no | 4 min |
| R1 | `r2_e2_v3` | E2 recipe + v3 annotation policy (CVV labels need contextual evidence) | yes | 15 min |
| R2 | `r2_e3a_hardneg` | R1 + hard-negative documents (experiment A) | yes | 17 min |
| R3 | `r2_e3b_positive` | R1 + difficult-positive documents (experiment B) | yes | 17 min |
| R4 | `r2_e4_combined` | R1 + A + B | yes | 19 min |
| D | | validate every artifact, write `round2_results.zip` | no | 1 min |

Completed stages are recorded in `OUT/status/<stage>.json` and skipped on re-run, so after a disconnect just
**Run all** again. The E2 checkpoint is only read, and its weight hash is checked before and after R0.

In [ ]:
# ---- paths: edit only if your Drive layout differs ------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/confidentiality_detector"
BUNDLE = f"{DRIVE_ROOT}/colab_bundle_round2.zip"                              # upload dist/colab_bundle_round2.zip here
E2_CKPT = f"{DRIVE_ROOT}/round1_rerun/checkpoints/e2_mbert_v2_fmtA"         # round-1 E2 checkpoint (read-only)
OUT = f"{DRIVE_ROOT}/round2"                                                # all round-2 outputs
WORK = "/content/cd2"
EXPECTED_BUNDLE_VERSION = "round2-r1"

## Setup 1/2 — mount Drive, check GPU, extract and verify the bundle

In [ ]:
import os, sys, shutil, zipfile, importlib
from google.colab import drive

drive.mount("/content/drive")
for label, path, is_file in [("bundle", BUNDLE, True), ("E2 checkpoint", E2_CKPT, False)]:
    if not (os.path.isfile(path) if is_file else os.path.isdir(path)):
        raise FileNotFoundError(f"{label} not found at {path}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: Runtime > Change runtime type > T4 GPU, then Run all again.")
print("GPU:", torch.cuda.get_device_name(0))

if os.path.isdir(WORK):
    shutil.rmtree(WORK)
with zipfile.ZipFile(BUNDLE) as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"corrupt bundle member: {bad}")
    zf.extractall(WORK)

sys.path.insert(0, f"{WORK}/scripts")
import colab_runner, colab_round2 as r2
importlib.reload(colab_runner); importlib.reload(r2)
if r2.BUNDLE_VERSION != EXPECTED_BUNDLE_VERSION:
    raise RuntimeError(f"bundle version {r2.BUNDLE_VERSION} != {EXPECTED_BUNDLE_VERSION}: upload the rebuilt bundle")
ctx = r2.Context(spec=r2.ROUND2, work=WORK, out=OUT, models={"e2": E2_CKPT})
os.makedirs(OUT, exist_ok=True)
r2.verify_bundle(ctx)

## Setup 2/2 — dependencies, import check in a fresh interpreter, unit tests

In [ ]:
r2.install_dependencies(ctx)
env_info = r2.check_environment(ctx, require_gpu=True)
r2.run_tests(ctx)
print("setup OK:", {k: env_info["versions"].get(k) for k in ("torch", "transformers", "seqeval", "phonenumbers", "pip")})

## R0 — baseline: existing E2 checkpoint on the v3 evaluation suite (no training)

In [ ]:
r2.run_stage(ctx, "R0")

## R1 — E2 recipe + v3 annotation policy

In [ ]:
r2.run_stage(ctx, "R1")

## R2 — experiment A: + hard negatives

In [ ]:
r2.run_stage(ctx, "R2")

## R3 — experiment B: + difficult positives

In [ ]:
r2.run_stage(ctx, "R3")

## R4 — A + B combined

In [ ]:
r2.run_stage(ctx, "R4")

## D — package results (predictions, metrics, logs, status, training histories; **no weights**)
Refuses unless every stage is `ok` and re-validates. Download `OUT/round2_results.zip`.

In [ ]:
archive = r2.package(ctx)
print("Send this file back:", archive)